In [ ]:
%matplotlib ipympl

In [ ]:
import functools
import pandas as pd
import numpy as np
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import scipy.interpolate as sci_interp
import scipy.optimize as sci_opt
import scipy.fft as sci_fft

jax.config.update("jax_enable_x64", True)

In [ ]:
def load_clean_references(file_path: str) -> tuple[jax.Array, jax.Array]:
    data = np.array(pd.read_hdf(file_path))
    return data[:, 1:4], data[:, 4:]

file_path = "/Users/jozbee/work/eng/comp/data/clean_specific-forces-standard-road-v2.hdf"
acc_ref, omega_ref = load_clean_references(file_path)
acc_ref = jnp.clip(acc_ref, -1.0, 1.0)

In [ ]:
data_size = 90 * 200
dt = 0.005
ts = np.arange(data_size) * dt
data = acc_ref[:data_size, 0]
ref_data = sci_interp.make_smoothing_spline(ts, data, lam=1e0)(ts)
ref_datap = sci_interp.make_smoothing_spline(ts, data, lam=1e0)(ts, nu=1)

## data filter helpers

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def get_E0_E1_C(den, num, K, nu=0):
    """Integration ZOH scheme."""
    a = jnp.poly(den)
    b = K * jnp.poly(num)
    b = jnp.concatenate([jnp.atleast_1d(b), jnp.zeros(nu)])
    assert a.size - b.size >= 1, f"(a, b) = ({a.size}, {b.size})"

    a_coeffs = a[1:]
    n = a_coeffs.size
    A = jnp.vstack([-a_coeffs, jnp.eye(n - 1, n)])
    B = jnp.zeros(n)
    B = B.at[0].set(1.)
    C = jnp.concatenate([jnp.zeros(n - b.size), b])

    Z = jnp.zeros_like(A)
    I = jnp.eye(*A.shape)  # noqa: E741
    dyn_mat = jnp.block([[A, Z], [I, Z]])
    y0 = jnp.block([[I], [Z]])
    E1 = (jax.scipy.linalg.expm(dyn_mat * dt) @ y0)[A.shape[0] :] @ B
    E0 = jax.scipy.linalg.expm(A * dt)
    return E0, E1, C

In [ ]:
@functools.partial(jax.jit, static_argnames=["nu"])
def linear_filt(den, num, K, data, x0=None, nu=0):
    E0, E1, C = get_E0_E1_C(den, num, K, nu)

    def filt_body(x0, u):
        x0 = E0 @ x0 + E1 * u
        y = C @ x0
        return x0, y
    
    if x0 is None:
        x0 = jnp.zeros(E0.shape[0])
    _, y = jax.lax.scan(filt_body, x0, data)
    return y

In [ ]:
f = 0.007676163090760176
f = -jnp.log(1 - f) * dt**-1
exp_den = jnp.array([-f])
exp_num = jnp.array([])
exp_K = f
# trip_exp_f = 2.0 * np.pi
trip_exp_f = 4.0
# trip_exp_den = jnp.array([-trip_exp_f, -trip_exp_f + 0.5, -trip_exp_f + 1.0])
trip_exp_den = jnp.array([-trip_exp_f, -trip_exp_f, -trip_exp_f])
# trip_exp_den = jnp.array([-trip_exp_f + 6.0])
trip_exp_num = jnp.array([-10.0])
trip_exp_K = jnp.prod(-trip_exp_den) / jnp.prod(-trip_exp_num)
exp_filt = linear_filt(exp_den, exp_num, exp_K, data)
trip_exp_filt = linear_filt(trip_exp_den, trip_exp_num, trip_exp_K, data)
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(data, label="data", alpha=0.2)
ax.plot(ref_data, label="ref_data", alpha=0.4)
ax.plot(exp_filt, label="exp_filt")
ax.plot(trip_exp_filt, label="trip_exp_filt")
ax.legend()
ax.grid()

## cost

In [ ]:
def decompose_params(params, den_n, num_n):
    den = -jnp.square(params[:den_n])
    num = -jnp.square(params[den_n: den_n + num_n])
    K = jnp.prod(-den) / jnp.prod(-num)
    return den, num, K

def lin_opt_cost(params, den_n, num_n, data, ref_data):
    assert den_n - num_n >= 1
    assert params.shape == (den_n + num_n,)
    den, num, K = decompose_params(params, den_n, num_n)
    data_filt = linear_filt(den, num, K, data)
    cost = jnp.mean(jnp.square(data_filt - ref_data)) * 1e2
    if den_n - num_n > 2:
        data_filtpp = linear_filt(den, num, K, data, nu=2)
        cost += jnp.mean(jnp.square(data_filtpp))
    return cost

lin_opt_cost_jit = jax.jit(lin_opt_cost, static_argnames=["den_n", "num_n"])
lin_opt_cost_grad = jax.jit(jax.value_and_grad(lin_opt_cost), static_argnames=["den_n", "num_n"])

In [ ]:
np.random.seed(67)
den_n = 4
num_n = 2
params0 = jnp.array(np.random.uniform(0.1, 2.0, den_n + num_n))
# params0 = jnp.sqrt(jnp.array([trip_f, trip_f + 0.1, trip_f + 0.2]))
# params0 = jnp.sqrt(jnp.array([f]))
res = sci_opt.minimize(
    fun=functools.partial(lin_opt_cost_grad, den_n=den_n, num_n=num_n, data=data, ref_data=ref_data),
    x0=params0,
    method="L-BFGS-B",
    jac=True,
)
params0, res

In [ ]:
params0, res.x

In [ ]:
lin_opt_cost_jit(params0, den_n, num_n, data, ref_data), lin_opt_cost_jit(res.x, den_n, num_n, data, ref_data)

In [ ]:
plot_data = acc_ref[:150 * 200, 0]
plot_ts = np.arange(plot_data.size) * dt
plot_ref_data = sci_interp.make_smoothing_spline(plot_ts, plot_data, lam=1e0)(plot_ts)

res_decomp = list(decompose_params(res.x, den_n, num_n))
opt_filt = linear_filt(*res_decomp, data=plot_data)
exp_filt = linear_filt(exp_den, exp_num, exp_K, plot_data)
trip_exp_filt = linear_filt(trip_exp_den, trip_exp_num, trip_exp_K, plot_data)

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(plot_data, label="plot_data", alpha=0.2)
ax.plot(plot_ref_data, label="plot_ref_data", alpha=0.4)
ax.plot(exp_filt, label="exp_filt")
ax.plot(trip_exp_filt, label="trip_exp_filt")
ax.plot(opt_filt, label="opt_filt")

# if den_n > 1:
    # pass
    # opt_filtp = linear_filt(*res_decomp, data=plot_data, nu=1)
    # ax.plot(ref_datap, label="ref_datap", alpha=0.4)
    # ax.plot(opt_filtp, label="opt_filtp")

ax.legend()
ax.grid()

In [ ]:
jnp.prod(-res_decomp[0]), jnp.prod(-res_decomp[1]), res_decomp